# Phase 2: Image Preprocessing Pipeline

**Goal:** Build, inspect, and benchmark an OpenCV image preprocessing pipeline that enhances prescription text legibility before passing to OCR engines.

| Stage | Operation | Purpose |
|---|---|---|
| **Grayscale** | `cv2.cvtColor` | Removes color noise, reducing dimension |
| **Resize** | `resize_long_side` | Rescales text to standard size (~1600px long side) |
| **Deskew** | `cv2.warpAffine` | Straightens slightly tilted phone photos |
| **Denoise** | `cv2.fastNlMeansDenoising` | Eliminates sensor speckles & grain |
| **Binarize** | `cv2.adaptiveThreshold` | Compensates for uneven lighting & shadows |

## 0. Setup Cell

In [ ]:
# --- SETUP CELL ---
import os, sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / ".git").exists() and root != root.parent:
    root = root.parent
if not (root / ".git").exists():
    root = Path.cwd().resolve()
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repo root:", root)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 1. Load Data Index & Helpers

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import src.data as D
import src.preprocess as P
import importlib
importlib.reload(D)
importlib.reload(P)

from src.data import DATA_ROOT, build_image_index, open_image, gold_records

syn_index = build_image_index(os.path.join(DATA_ROOT, "synthetic"))
dev_gold_path = "data/dev_gold.csv"
if os.path.exists(dev_gold_path):
    gold_dev = gold_records(pd.read_csv(dev_gold_path))
    dev_files = sorted(gold_dev.keys())
    print(f"Loaded {len(dev_files)} dev images from {dev_gold_path}")
else:
    print("dev_gold.csv not generated yet. Using images found in synthetic index directly.")
    dev_files = sorted(syn_index.keys())[:20]
print("Sample images available:", len(dev_files))

## 2. Visual Comparison of Preprocessing Stages
Displays side-by-side: `gray`, `resized`, `denoised`, and `binary`.

In [ ]:
def show_stages(pil_img):
    st = P.preprocess(pil_img)
    names = ["gray", "resized", "denoised", "binary"]
    fig, axes = plt.subplots(1, 4, figsize=(20, 6))
    for ax, n in zip(axes, names):
        ax.imshow(st[n], cmap="gray")
        ax.set_title(n, fontsize=12)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

num_samples = min(3, len(dev_files))
for i in range(num_samples):
    fn = dev_files[i]
    print(f"Displaying stages for: {fn}")
    show_stages(open_image(fn, syn_index))

## 3. Benchmarking Preprocessing Speed
Ensure total preprocessing time per image is under ~1 to 2 seconds.

In [ ]:
num_test = min(10, len(dev_files))
if num_test > 0:
    t0 = time.time()
    for i in range(num_test):
        img = open_image(dev_files[i], syn_index)
        _ = P.preprocess(img)
    elapsed = time.time() - t0
    avg_secs = elapsed / num_test
    print(f"Processed {num_test} images in {elapsed:.2f} s. Average time: {avg_secs:.3f} s/image")
    assert avg_secs < 3.0, "Warning: Preprocessing exceeds speed budget; consider cv2.medianBlur(gray, 3) if needed."